# Cell 1 — Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import pickle
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Cell 2 — Device

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


# Cell 3 — Load Dataset

In [3]:
with open("../data/processed/dl/dl_train_test_split.pkl", "rb") as f:
    data = pickle.load(f)

X_train = data["X_train"]
X_test = data["X_test"]

y_train = data["y_train"]
y_test = data["y_test"]

# Cell 4 — Feature Scaling (Required by Paper)

In [4]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Cell 5 — Encode Labels

In [5]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)

print("Classes:", num_classes)

Classes: 15


# Cell 6 — Convert to Tensor

In [6]:
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

y_train = torch.tensor(y_train, dtype=torch.long).to(device)
y_test = torch.tensor(y_test, dtype=torch.long).to(device)

# Cell 7 — DataLoader

In [7]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True
)

# Cell 8 — DNN Architecture (Simple Feed-Forward)

The dataset paper describes a **simple shallow DNN** that converges quickly.

In [8]:
class DNNModel(nn.Module):

    def __init__(self, input_dim, num_classes):

        super(DNNModel, self).__init__()

        self.model = nn.Sequential(

            nn.Linear(input_dim, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

# Cell 9 — Initialize Model

In [9]:
input_dim = X_train.shape[1]

model = DNNModel(input_dim, num_classes).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

# Cell 10 — Training

In [10]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss}")

Epoch 1/10 Loss: 418.84114626795053
Epoch 2/10 Loss: 314.8380651026964
Epoch 3/10 Loss: 307.8740040771663
Epoch 4/10 Loss: 303.71679174900055
Epoch 5/10 Loss: 302.76562797278166
Epoch 6/10 Loss: 299.2846387885511
Epoch 7/10 Loss: 299.2513551712036
Epoch 8/10 Loss: 298.88144813105464
Epoch 9/10 Loss: 297.3705538585782
Epoch 10/10 Loss: 296.7023467682302


# Cell 11 — Evaluation

In [11]:
model.eval()

with torch.no_grad():

    outputs = model(X_test)

    _, predicted = torch.max(outputs, 1)

y_pred = predicted.cpu().numpy()
y_true = y_test.cpu().numpy()

# Cell 12 — Accuracy

In [12]:
accuracy = accuracy_score(y_true, y_pred)

print("DNN Accuracy:", accuracy)

DNN Accuracy: 0.9521525755304473


# Cell 13 — Classification Report

In [13]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.96      0.96      4806
           1       0.93      0.68      0.78      9871
           2       0.99      1.00      1.00     13588
           3       0.99      1.00      0.99     10013
           4       1.00      1.00      1.00     24314
           5       1.00      0.20      0.33       171
           6       1.00      1.00      1.00        72
           7       1.00      1.00      1.00    275611
           8       0.92      0.18      0.30      9987
           9       0.98      0.99      0.98      3996
          10       0.98      0.97      0.98      1938
          11       0.43      0.91      0.59     10165
          12       0.58      0.47      0.52      7361
          13       0.99      0.84      0.91     10005
          14       0.51      0.92      0.65      3013

    accuracy                           0.95    384911
   macro avg       0.89      0.81      0.80    384911
weighted avg       0.97   

# Cell 14 — Save Model

In [14]:
torch.save(
    model.state_dict(),
    "../models/dl/dnn_model.pth"
)